### dash-cytoscape


In [1]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [2]:
root0 = Path('/home/flavio/uv/perturb_agent')
root0_data = root0 / 'data'
root_colab = root0_data / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-ACC'

disease = PSI_ID

root_project = create_dir(root0_data, PROG_ID)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={ptw_min_num_of_degs_cut}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC/config/all_lfc_cutoffs_TCGA-ACC.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [3]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, 
            project=project, s_project=s_project, 
            root0=root0, root0_data=root0_data,
            prog_id=PROG_ID, psi_id=PSI_ID,
            case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
            std_filename=std_filename, std_filename_list=std_filename_list,
            geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
            tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
            LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
            num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
            min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
            saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(enr.echo_parameters())


Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-ACC
>>> Tumor


In [4]:
enr.dflfc_ori.head(2)

,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000234795,RFTN1P1,transcribed_unprocessed_pseudogene,26.457,2.951,8.966,3.086e-19,4.845e-18,121.435,DESeq2,26.457
1,ENSG00000138653,NDST4,protein_coding,23.413,2.952,7.930,2.189e-15,2.723e-14,13.755,DESeq2,23.413


#### Open DASH_CYTO passin dflfc_ori

In [5]:
dcy = DASH_CYTO(root0=ROOT0, root0_data=root0_data, dflfc_ori=enr.dflfc_ori)

In [6]:
dcy.root_ncbi

PosixPath('/home/flavio/uv/perturb_agent/data/colab/ncbi')

### fname_gene_alias = "hugo_gene_alias_table.tsv"

In [7]:
force=False
verbose=False

dfh = pdreadcsv(dcy.fname_hugo, dcy.root_ncbi, verbose=verbose)
dfh.columns

Index(['ensembl_id', 'symbol', 'name', 'uniprot_id', 'ncbi_gene_id', 'synonyms', 'refseq_summary'], dtype='object')

In [8]:
owl_file = "R-HSA-165159_level3.owl"
owl_file = "R-HSA-114608_level3.owl"

pathway = 'Platelet degranulation'
pathway_id = "R-HSA-114608"

pathway = 'Cell Cycle Checkpoints'
pathway_id = "R-HSA-69620"

pathway = 'Condensation of Prometaphase Chromosomes'
pathway_id = "R-HSA-2514853"

ret = dcy.read_owl(pathway_id, pathway)
print(ret)
rdf = dcy.rdf


True


```Python
G.nodes[gene_id]["log2FC"] = log2fc
G.nodes[gene_id]["FDR"] = fdr
G.nodes[gene_id]["mutated"] = True
```

### OWL to Graph

In [9]:
saved_positions = dcy.load_positions(pathway_id)

Data loaded from /home/flavio/uv/perturb_agent/data/colab/owl/positions_R-HSA-2514853.json


In [10]:
i=0
for key, posi in saved_positions.items():
    print(key, posi)
    i+=1
    if i==3: break

Pathway1 {'x': 1002.4966136962756, 'y': 145.77277320250457}
BiochemicalReaction1 {'x': 866.187921289478, 'y': 409.7718020225718}
BiochemicalReaction2 {'x': 1063.0649200090636, 'y': 376.69147888086735}


### Synonyms table

In [11]:
# df = pdreadcsv(dcy.fname_hugo, dcy.root_ncbi)
# df.head(2)

In [12]:
force=False
verbose=True

# dfsyn = dcy.build_gene_alias_table(force=force, verbose=verbose)
# dfsyn.head(3)

### Dash

In [ ]:
height = "95%"
width = "100%"
marginTop="20px"

dcy.run_app(height=height, width=width, marginTop=marginTop, port=8051)

Data loaded from /home/flavio/uv/perturb_agent/data/colab/owl/positions_R-HSA-2514853.json
>>> create_cytoscape_app()  Condensation of Prometaphase Chromosomes - R-HSA-2514853


TypeError: The `html.Div` component (version 2.18.2) with the ID "Div(children=[Button(children='1️⃣ Select 1st neighbors', id='btn-select-neighbors', className='popup-button'), Button(children='⬆️ Select upstream', id='btn-select-upstream', className='popup-button'), Button(children='⬇️ Select downstream', id='btn-select-downstream', className='popup-button'), Button(children='⭐ Select main hubs', id='btn-select-hubs', className='popup-button'), Button(children='🔼 Select most upstream nodes', id='btn-select-sources', className='popup-button'), Button(children='🔽 Select most downstream nodes', id='btn-select-sinks', className='popup-button'), Button(children='✖ Close', id='btn-close-popup', className='popup-button popup-close')], id='cyto-popup-menu', style={'display': 'none', 'position': 'absolute', 'zIndex': 9999, 'backgroundColor': 'white', 'border': '1px solid #ddd', 'borderRadius': '12px', 'boxShadow': '0 8px 24px rgba(0,0,0,0.18)', 'padding': '10px', 'width': '230px'})" detected 